# **📊 Clean & Jerk Dataset — Sync & Summary Explorer**

### 🔧 Colab Setup & Library Init

In [ ]:
# @title 🔧 Colab Setup & Library Init { display-mode: "form" }
import importlib, sys, types
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp = types.ModuleType("imp")
        _imp.reload = importlib.reload
        sys.modules["imp"] = _imp

%load_ext autoreload
%autoreload 2

import importlib.util, os, subprocess

# 1. Colab-specific: mount Google Drive & configure renderers
try:
    from google.colab import drive, data_table
    import plotly.io as pio
    plotly_renderer = "colab" # @param ["colab", "vscode", "notebook_connected", "iframe"]
    pio.renderers.default = plotly_renderer
    data_table.enable_dataframe_formatter()
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print(f"⚠️ Drive not mounted ({e}). Mount via Colab sidebar if needed.")
except ImportError:
    pass

# 2. Ensure dependencies are installed
if importlib.util.find_spec("h5py") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "h5py"])

# 3. Clone (or pull) public repo into /content/Wearable-Biomechanics and add to sys.path
repo_dir = "/content/Wearable-Biomechanics" # @param {type:"string"}
git_branch = "dataset_explorer" # @param {type:"string"}

if os.path.exists(os.path.join(".", "src", "__init__.py")):
    repo_root = os.path.abspath(".")
else:
    if not os.path.exists(repo_dir):
        subprocess.check_call([
            "git", "clone", "-b", git_branch,
            "https://github.com/janithcyapa/Wearable-Biomechanics.git", repo_dir
        ])
    else:
        subprocess.run(["git", "-C", repo_dir, "checkout", git_branch], check=False)
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=False)
        for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
            importlib.reload(sys.modules[_m])
    repo_root = os.path.abspath(repo_dir)

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src import display_table
from src.explorer import DEFAULT_CLEAN_DATASET_DIR, load_summary_tables, load_set_h5
from scripts.sync_summaries_to_h5 import sync_dataset_h5

dataset_dir = DEFAULT_CLEAN_DATASET_DIR # @param {type:"string"}
print(f"📂 Dataset folder: {dataset_dir}")

### 🔄 Sync Summary CSVs → `.h5` & Remove `/fault_log`

Run with `dry_run = True` first to preview counts, then uncheck `dry_run` to apply updates to all `sets/*.h5` files.

In [ ]:
# @title 🔄 Sync Summary CSVs to sets/*.h5 & Remove /fault_log { display-mode: "form" }
dry_run = True # @param {type:"boolean"}
remove_fault_log = True # @param {type:"boolean"}
sync_sets = True # @param {type:"boolean"}
sync_participants = True # @param {type:"boolean"}

sync_stats = sync_dataset_h5(
    dataset_dir=dataset_dir,
    remove_fault_log=remove_fault_log,
    sync_sets=sync_sets,
    sync_participants=sync_participants,
    dry_run=dry_run,
)

### 📁 Root Summary DataFrames

In [ ]:
# @title 📁 Load All Root Summary Tables
tables = load_summary_tables(dataset_dir)
for name, df in tables.items():
    if not df.empty:
        print(f"• {name}: {df.shape[0]} rows × {df.shape[1]} cols")

In [ ]:
# @title 👥 Participants Summary (`participant_summary.csv`)
display_table(tables["participants"], clean_columns=False)

In [ ]:
# @title 🏋️ Sets Summary (`sets_summary.csv`)
display_table(tables["sets"], clean_columns=False)

In [ ]:
# @title 📡 26-IMU Device Summary (`device_summary.csv`)
display_table(tables["devices"], clean_columns=False)

In [ ]:
# @title ⏱️ Phase Markers Summary (`phase_markers_summary.csv`)
display_table(tables["phase_markers"], clean_columns=False)

In [ ]:
# @title 📖 Data Dictionary (`data_dictionary.csv`)
display_table(tables["data_dictionary"], clean_columns=False)

### 📦 Inspect Per-Set `.h5` File (Metadata, Participant Metadata & DataFrames)

In [ ]:
# @title 📦 Load Set HDF5 & Inspect Structure { display-mode: "form" }
import h5py
import pandas as pd

set_id = "set_0001" # @param {type:"string"}
sd = load_set_h5(dataset_dir, set_id)

with h5py.File(sd["h5_path"], "r") as f:
    print("HDF5 top-level groups:", list(f.keys()))

print(f"accel shape: {sd['accel'].shape} | gyro shape: {sd['gyro'].shape} | time_s: {sd['time_s'].shape}")
print(f"sensors ({len(sd['sensors'])}): {sd['sensors']}")

In [ ]:
# @title 📋 Set `/metadata` & `/participant_metadata` Attributes
print("--- /metadata ---")
display_table(pd.DataFrame(list(sd["metadata"].items()), columns=["attribute", "value"]), clean_columns=False)

print("--- /participant_metadata ---")
display_table(pd.DataFrame(list(sd["participant_metadata"].items()), columns=["attribute", "value"]), clean_columns=False)

In [ ]:
# @title 📈 IMU DataFrame (`sd["imu_df"]` — first 10 rows)
display_table(sd["imu_df"].head(10), clean_columns=False)

In [ ]:
# @title ⏱️ Phase Markers — Consensus (`sd["consensus_df"]`) & Annotators (`sd["annotators_df"]`)
print("--- consensus_df ---")
display_table(sd["consensus_df"], clean_columns=False)

print("--- annotators_df (first 20 rows) ---")
display_table(sd["annotators_df"].head(20), clean_columns=False)